# MEERA API Backend Manual Testing Notebook

This notebook contains cells to make HTTP requests using the `requests` library to test all endpoints of the **MEERA API Backend**.

### Prerequisite:
Make sure your FastAPI server is running. You can start it locally using:
```bash
uv run uvicorn src.main:app --reload
```

In [2]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"
print(f"Target API Base URL: {BASE_URL}")

Target API Base URL: http://127.0.0.1:8000


## 1. Health Check

Check if the server is up and healthy.

In [4]:
response = requests.get(f"{BASE_URL}/")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "status": "healthy",
  "service": "MEERA Backend",
  "version": "1.0.0"
}


## 2. Department Mapping Master

Upload a department mapping CSV/Excel parsed json payload and fetch it.

In [5]:
# Upload Master file payload
upload_payload = {
    "records": [
        {
            "office": "Revenue Central Office",
            "division_section": "Property Taxes",
            "sub_section": "Zone A",
            "user": "revenue_officer_1"
        },
        {
            "office": "Revenue Central Office",
            "division_section": "Commercial Taxes",
            "sub_section": "Zone B",
            "user": "revenue_officer_2"
        }
    ],
    "department": "revenue"
}

response = requests.post(f"{BASE_URL}/api/v1/upload/department-mapping-master", json=upload_payload)
print(f"Upload Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Upload Status Code: 200
{
  "data": [
    {
      "upload_status": "success",
      "total_records_uploaded": "2"
    }
  ],
  "message": "Department master uploaded successfully",
  "error": null
}


In [6]:
# Fetch Master data for the department
response = requests.get(f"{BASE_URL}/api/v1/department-mapping-master", params={"department": "revenue"})
print(f"Get Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Get Status Code: 200
{
  "data": {
    "records": [
      {
        "office": "Revenue Central Office",
        "division_section": "Property Taxes",
        "sub_section": "Zone A",
        "department": "revenue",
        "user": "revenue_officer_1"
      },
      {
        "office": "Revenue Central Office",
        "division_section": "Commercial Taxes",
        "sub_section": "Zone B",
        "department": "revenue",
        "user": "revenue_officer_2"
      }
    ]
  },
  "message": "Department master retrieved successfully",
  "error": null
}


## 3. RTI Queries List & Filtering

Retrieve all RTI query records, optionally filtered by status, assignment parameters, or pagination.

In [7]:
# List all queries (initially empty if database was clean, or populated from database startup)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "data": [],
  "message": "Queries fetched successfully",
  "error": null
}


In [8]:
# Fetch using validation constraints: Combined filters (should return 400 Bad Request)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries", params={"unassigned_only": "true", "assigned_to": "officer_1"})
print(f"Status Code (Expected 400): {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code (Expected 400): 400
{
  "data": [],
  "message": "Cannot combine assigned_to filter with unassigned_only",
  "error": "INVALID_FILTER_COMBINATION"
}


## 4. RTI Query Aggregate Counts

Returns global aggregate counts grouped by status.

In [9]:
response = requests.get(f"{BASE_URL}/api/v1/rti-queries/count")
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "data": {
    "total_count": 0,
    "pending_count": 0,
    "resolved_count": 0
  },
  "message": "Counts retrieved successfully",
  "error": null
}


## 5. RTI Query Detail & Correspondence

Fetches detailed view of a single query including supporting documents and office notes (sorted newest first).

In [10]:
# Test Detail lookup for a non-existent ID (should return 404)
response = requests.get(f"{BASE_URL}/api/v1/rti-queries/not_found")
print(f"Status Code (Expected 404): {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code (Expected 404): 404
{
  "data": null,
  "message": "RTI query not_found not found",
  "error": "RTI_QUERY_NOT_FOUND"
}


## 6. Chat & FAQ Deflection Interaction

Submit citizen/officer queries against an RTI query, fetch assistant suggestions, and download the conversation log.

In [11]:
# POST a user query
chat_payload = {
    "user_query": "Can we request an extension for zone property audit?",
    "user_id": "officer_1",
    "rti_query_id": "query_123"  # Make sure this ID is populated in your DB or matches
}

response = requests.post(f"{BASE_URL}/api/v1/user_query", json=chat_payload)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "data": [
    {
      "query_id": "cf446e4c-b0d0-435a-82c8-b51e082a5a2e",
      "asst_response": "Mock response for user query.",
      "source": [
        "source_document_1.pdf"
      ]
    }
  ],
  "message": "Query processed successfully",
  "error": null
}


In [12]:
# POST to request assistant suggestion
sug_payload = {
    "rti_query_id": "query_123",
    "user_id": "officer_1"
}
response = requests.post(f"{BASE_URL}/api/v1/get-suggestion", json=sug_payload)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "data": [
    {
      "suggestion_id": "mock_suggestion_uuid",
      "asst_suggestion": "Mock suggested resolution steps.",
      "source": [
        "source_guideline_2.pdf"
      ]
    }
  ],
  "message": "Suggestion retrieved successfully",
  "error": null
}


In [13]:
# GET current chat session and suggestions
response = requests.get(
    f"{BASE_URL}/api/v1/get-session",
    params={"rti_query_id": "query_123", "user_id": "officer_1"}
)
print(f"Status Code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status Code: 200
{
  "data": {
    "rti_query_id": "query_123",
    "asst_suggestion": "Mock suggested resolution steps.",
    "chat": [
      {
        "user_query": "Can we request an extension for zone property audit?",
        "user_query_id": "cf446e4c-b0d0-435a-82c8-b51e082a5a2e",
        "source": "source_document_1.pdf",
        "asst_response": "Mock response for user query."
      }
    ]
  },
  "message": "Session retrieved successfully",
  "error": null
}
